# 06. Task Completion & Budgets — deep dive

Notebook 01 (`arcrun/01-core-react`) introduces the agentic loop and shows `task_complete` and `max_turns` in passing. This notebook is the deep dive on the same surface: every status code, every field of `TaskCompleteArgs`, exactly what the loop emits when budgets breach, and the patterns an outer caller (e.g. `arcagent`) uses to consume the structured completion payload.

**Why this matters.** The difference between *the agent finished* and *the agent gave up* is not a subjective judgement — it is a structured signal. Without it, downstream code has to guess from text ("did Claude really do the thing?"), which is exactly the kind of fragile reasoning you do **not** want at the boundary between an agent and the rest of your system. arcrun's loop guarantees a single, machine-readable terminator on every run, even when the run is killed by a budget cap.

**You will learn:**

- The full schema of `TaskCompleteArgs` (status, summary, artifacts, next_steps, error)
- The semantics of every status: `success`, `partial`, `failed`
- How the loop enforces `max_turns` and `max_cost_usd`, and what payload it synthesises
- The `make_budget_breach_args` helper — the canonical breach vocabulary
- The `signals_completion=True` mechanism that makes `task_complete` work without the loop knowing its name
- The `loop.completed` event and its shape
- How `task_complete` interacts with both ReAct and CodeExec strategies
- Patterns for an outer caller to branch on the payload (retry, hand-off, escalation)

Everything in this notebook runs **without an API key** — we drive the loop with a tiny
`StubModel` that emits canned responses, exactly the way arcrun's own tests do.

## 1. Setup

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Now pull in the symbols this notebook touches. Every import below points at a public surface — these are the names downstream code is supposed to use.

In [2]:
from arcrun.builtins.task_complete import (
    TaskCompleteArgs,
    TaskStatus,
    make_budget_breach_args,
    make_task_complete_tool,
)
from arcrun.events import EventBus
from arcrun.registry import ToolRegistry
from arcrun.sandbox import Sandbox
from arcrun.state import RunState
from arcrun.strategies.code import CodeExecStrategy
from arcrun.strategies.react import react_loop
from arcrun.types import LoopResult, SandboxConfig, Tool

print("TaskCompleteArgs:", TaskCompleteArgs)
print("TaskStatus literal:", TaskStatus)
print("make_task_complete_tool:", make_task_complete_tool)
print("make_budget_breach_args:", make_budget_breach_args)

TaskCompleteArgs: <class 'arcrun.builtins.task_complete.TaskCompleteArgs'>
TaskStatus literal: typing.Literal['success', 'partial', 'failed']
make_task_complete_tool: <function make_task_complete_tool at 0x10a443100>
make_budget_breach_args: <function make_budget_breach_args at 0x10a4431a0>


## 2. Why structured completion

Imagine you wired up an agent to draft a customer email. The loop returns a string. Did the agent:

1. Finish the email and signal that it's done? **success**
2. Write the first paragraph and stop because it ran out of context? **partial**
3. Try the customer-lookup tool, see permission-denied, and bail? **failed**
4. Get killed by the loop because it spun for 25 turns? **budget breach**

All four cases produce some text. None of the four can be reliably told apart by inspecting that text.

The fix in arcrun is small and explicit: there is one tool — `task_complete` — flagged with
`signals_completion=True`. When the model invokes it, the loop terminates and stores the call's
*arguments* on the run state as `completion_payload`. That dict is the source of truth. Budget caps
do the same — the loop synthesises a `task_complete(failed, error=...)` payload itself when it
kills the run.

Result: every run ends with a structured payload. The outer caller branches on `status` and `error`,
not on a substring of free-form text.

In [3]:
# The mechanism that makes this work, in one line:
tool = make_task_complete_tool()
print(f"name:                {tool.name}")
print(f"signals_completion:  {tool.signals_completion}")
print(f"timeout_seconds:     {tool.timeout_seconds}")
print()
print("When the loop sees a tool_call whose registered Tool has signals_completion=True,")
print("it stops. The arguments become state.completion_payload.")

name:                task_complete
signals_completion:  True
timeout_seconds:     5.0

When the loop sees a tool_call whose registered Tool has signals_completion=True,
it stops. The arguments become state.completion_payload.


## 3. `TaskCompleteArgs` — the schema in full

Every field, every validation rule. `TaskCompleteArgs` is a frozen Pydantic model — an instance is
immutable once constructed.

| Field        | Type                | Required | Purpose                                                  |
|--------------|---------------------|----------|----------------------------------------------------------|
| `status`     | `Literal[...]`      | yes      | One of `success`, `partial`, `failed`.                   |
| `summary`    | `str`               | yes      | One-line human-readable summary.                         |
| `artifacts`  | `list[str] \| None` | no       | File paths or URIs the agent produced.                   |
| `next_steps` | `list[str] \| None` | no       | Follow-up actions for the user / next agent.             |
| `error`      | `str \| None`       | no       | Structured error code (`max_turns`, `max_cost`, etc).    |

`TaskStatus` is exported as a `Literal` type so static checkers can prove your code only branches
on the three legal values.

In [4]:
# Minimal valid payload
args = TaskCompleteArgs(status="success", summary="wrote 3 docs")
print(args)
print()
print("artifacts default :", args.artifacts)
print("next_steps default:", args.next_steps)
print("error default     :", args.error)

status='success' summary='wrote 3 docs' artifacts=None next_steps=None error=None

artifacts default : None
next_steps default: None
error default     : None


In [5]:
# Full payload: every optional field set
full = TaskCompleteArgs(
    status="partial",
    summary="completed 2 of 3 steps",
    artifacts=["file:///workspace/report.md", "file:///workspace/summary.txt"],
    next_steps=["re-run the third step with smaller batch size"],
    error=None,
)
for field, value in full.model_dump().items():
    print(f"  {field:11s}: {value!r}")

  status     : 'partial'
  summary    : 'completed 2 of 3 steps'
  artifacts  : ['file:///workspace/report.md', 'file:///workspace/summary.txt']
  next_steps : ['re-run the third step with smaller batch size']
  error      : None


In [6]:
# Frozen: instances cannot be mutated after construction.
from pydantic import ValidationError

try:
    full.status = "success"  # type: ignore[misc]
except ValidationError as exc:
    print("Mutation blocked:")
    print(f"  {exc.errors()[0]['msg']}")

Mutation blocked:
  Instance is frozen


In [7]:
# status is restricted to the three literal values. Anything else is a ValidationError.
try:
    TaskCompleteArgs(status="kinda_done", summary="x")  # type: ignore[arg-type]
except ValidationError as exc:
    err = exc.errors()[0]
    print(f"type    : {err['type']}")
    print(f"loc     : {err['loc']}")
    print(f"message : {err['msg']}")

type    : literal_error
loc     : ('status',)
message : Input should be 'success', 'partial' or 'failed'


In [8]:
# summary is required — the agent must say something about what happened.
try:
    TaskCompleteArgs(status="success")  # type: ignore[call-arg]
except ValidationError as exc:
    print("summary required:")
    for e in exc.errors():
        print(f"  {e['loc']}: {e['msg']}")

summary required:
  ('summary',): Field required


Now wire up the stubs we need to drive the loop end-to-end. These mirror what arcrun's own
`tests/test_loop_termination.py` uses — minimal, deterministic, mock-only.

In [9]:
# A tiny model that replays a fixed sequence of responses, plus matching response/tool-call types.
from dataclasses import dataclass, field
from typing import Any


@dataclass
class StubToolCall:
    id: str
    name: str
    arguments: dict[str, Any]


@dataclass
class StubUsage:
    input_tokens: int = 5
    output_tokens: int = 3
    total_tokens: int = 8


@dataclass
class StubResponse:
    content: str = ""
    stop_reason: str = "tool_use"
    tool_calls: list[StubToolCall] = field(default_factory=list)
    cost_usd: float = 0.0
    usage: StubUsage = field(default_factory=StubUsage)


class StubModel:
    def __init__(self, responses: list[StubResponse]) -> None:
        self._responses = list(responses)
        self._idx = 0

    async def invoke(self, _messages, tools=None, **_kwargs):
        if self._idx >= len(self._responses):
            # Fall back to end_turn so under-specified scripts terminate cleanly.
            return StubResponse(stop_reason="end_turn")
        resp = self._responses[self._idx]
        self._idx += 1
        return resp


def make_run_state(
    *, max_cost_usd: float | None = None, extra_tools: list[Tool] | None = None
) -> RunState:
    bus = EventBus(run_id="nb-06")
    tools: list[Tool] = [make_task_complete_tool()]
    if extra_tools:
        tools.extend(extra_tools)
    registry = ToolRegistry(tools, bus)
    return RunState(
        messages=[],
        registry=registry,
        event_bus=bus,
        run_id="nb-06",
        max_cost_usd=max_cost_usd,
    )


async def run_with(state: RunState, model: StubModel, *, max_turns: int = 10) -> LoopResult:
    sandbox = Sandbox(SandboxConfig(), state.event_bus)
    return await react_loop(model, state, sandbox, max_turns=max_turns)


print("stubs ready:", StubModel, StubResponse, StubToolCall)

stubs ready: <class '__main__.StubModel'> <class '__main__.StubResponse'> <class '__main__.StubToolCall'>


## 4. `status: success` — the happy path

The model calls `task_complete(status='success', summary=...)` once it believes the user's request
has been satisfied. The loop:

1. Sees a tool_call to a registered tool with `signals_completion=True`.
2. Stores the call's arguments on `state.completion_payload`.
3. Emits a `loop.completed` event with the same payload.
4. Returns a `LoopResult` whose `content` is the *summary* of the payload.

Crucially, the loop has **no hard-coded knowledge of the name `task_complete`**. The mechanism is
the `signals_completion` flag on the `Tool`. That means you can ship your own structured terminator
tools (with different schemas — e.g. an audit-required terminator) by setting the same flag.

In [10]:
model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            tool_calls=[
                StubToolCall(
                    id="tc-1",
                    name="task_complete",
                    arguments={
                        "status": "success",
                        "summary": "drafted the welcome email",
                        "artifacts": ["file:///workspace/welcome.md"],
                    },
                )
            ],
        ),
    ]
)

state = make_run_state()
result = await run_with(state, model)

print("completion_payload:", state.completion_payload)
print()
print("LoopResult.content :", result.content)
print("LoopResult.turns   :", result.turns)
print("strategy_used      :", result.strategy_used)

completion_payload: {'status': 'success', 'summary': 'drafted the welcome email', 'artifacts': ['file:///workspace/welcome.md']}

LoopResult.content : drafted the welcome email
LoopResult.turns   : 1
strategy_used      : react


In [11]:
# The loop also emits a loop.completed event whose data field is the payload.
completion_events = [e for e in state.event_bus.events if e.type == "loop.completed"]
print(f"loop.completed events: {len(completion_events)}")
for evt in completion_events:
    print()
    print(f"  type      : {evt.type}")
    print(f"  run_id    : {evt.run_id}")
    print("  data:")
    for k, v in evt.data.items():
        print(f"    {k:9s}: {v!r}")

loop.completed events: 1

  type      : loop.completed
  run_id    : nb-06
  data:
    status   : 'success'
    summary  : 'drafted the welcome email'
    artifacts: ['file:///workspace/welcome.md']


Notice three things:

- The agent's tool_call **arguments** become the payload, verbatim.
- `LoopResult.content` is the *summary*, not the full payload — because callers that only render
  text (CLIs, logs) don't need the full structured form.
- The full structured form is on `state.completion_payload` and on the `loop.completed` event.
  Outer code reads from there.

## 5. `status: partial` — "I did some of it"

`partial` is the most useful status in production. It says: *the run terminated cleanly, but the
user's task is not fully done*. The agent uses it when it wants to hand back work in progress —
say, three of five sub-tasks completed, or one section of a report drafted but the data for the
remaining sections requires a tool the agent doesn't have.

`partial` is the right place for `next_steps` — explicit, machine-readable instructions for what
the next caller (or human) should do.

In [12]:
model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            tool_calls=[
                StubToolCall(
                    id="tc-1",
                    name="task_complete",
                    arguments={
                        "status": "partial",
                        "summary": "drafted 2 of 3 sections; sales data unavailable",
                        "artifacts": ["file:///workspace/report-draft.md"],
                        "next_steps": [
                            "fetch Q3 sales data from the data warehouse",
                            "re-invoke this agent with section_3_data populated",
                        ],
                    },
                )
            ],
        ),
    ]
)

state = make_run_state()
result = await run_with(state, model)

payload = state.completion_payload
print(f"status     : {payload['status']}")
print(f"summary    : {payload['summary']}")
print(f"artifacts  : {payload['artifacts']}")
print("next_steps :")
for step in payload["next_steps"]:
    print(f"  - {step}")

status     : partial
summary    : drafted 2 of 3 sections; sales data unavailable
artifacts  : ['file:///workspace/report-draft.md']
next_steps :
  - fetch Q3 sales data from the data warehouse
  - re-invoke this agent with section_3_data populated


An outer caller branches on `status` to decide what to do. The shape below is the canonical
pattern — we'll come back to it in section 11.

In [13]:
def render_handoff(payload: dict) -> str:
    """Mock outer-caller logic that turns a partial payload into a hand-off message."""
    if payload["status"] != "partial":
        return f"(not a partial outcome — status={payload['status']})"
    lines = [f"TASK INCOMPLETE: {payload['summary']}"]
    if payload.get("artifacts"):
        lines.append(f"  artifacts produced: {payload['artifacts']}")
    if payload.get("next_steps"):
        lines.append("  follow up:")
        for step in payload["next_steps"]:
            lines.append(f"    - {step}")
    return "\n".join(lines)


print(render_handoff(state.completion_payload))

TASK INCOMPLETE: drafted 2 of 3 sections; sales data unavailable
  artifacts produced: ['file:///workspace/report-draft.md']
  follow up:
    - fetch Q3 sales data from the data warehouse
    - re-invoke this agent with section_3_data populated


## 6. `status: failed` — the model signals failure

`failed` is for the case where the agent itself decides the task cannot be completed. Common
examples: a required tool returned permission-denied, the input is malformed and the agent
concluded it cannot recover, or an external dependency is unavailable.

The `error` field is for **structured** error codes — short tokens an outer caller can switch on.
Free-form prose belongs in `summary`.

In [14]:
model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            tool_calls=[
                StubToolCall(
                    id="tc-1",
                    name="task_complete",
                    arguments={
                        "status": "failed",
                        "summary": "customer-lookup tool returned permission_denied",
                        "error": "tool_permission_denied",
                    },
                )
            ],
        ),
    ]
)

state = make_run_state()
result = await run_with(state, model)
payload = state.completion_payload

print(f"status   : {payload['status']}")
print(f"summary  : {payload['summary']}")
print(f"error    : {payload['error']}")
print()
print("LoopResult.turns         :", result.turns)
print("LoopResult.tool_calls_made:", result.tool_calls_made)

status   : failed
summary  : customer-lookup tool returned permission_denied
error    : tool_permission_denied

LoopResult.turns         : 1
LoopResult.tool_calls_made: 1


An outer caller distinguishes *agent-signalled failure* (status=failed) from *budget breach*
(status=failed, error=`max_turns`/`max_cost`) by reading `error`. Both share the same `failed`
status — that's deliberate, so a generic "did the run fail?" check is a single equality on
`status`. Specialised handling reads `error`.

## 7. `max_turns` enforcement

Every call to `run()` accepts `max_turns` (default 25). The unified top-of-turn breaker
(`check_breaker`) halts the moment `turn_count >= max_turns`. If the model never calls
`task_complete` and burns through the limit, the loop:

1. Sets `state.completion_payload = {status: 'failed', summary: 'Turn limit reached...', error: 'max_turns'}`.
2. Emits `loop.max_turns` with `turns_used` and `max_turns`.
3. Emits `loop.completed` with `{reason, cost_usd, tokens}` — the reason token, not the full payload (the synthetic payload lives on `state.completion_payload`).
4. Returns a `LoopResult` with `content=None`.

The contract from the outer caller's side is identical: the run ended with a structured payload.
There is no special case to write.

In [15]:
# A model that loops forever — keeps calling a tool that does nothing useful.
import itertools

_tc_counter = itertools.count()


class NeverFinishesModel:
    async def invoke(self, _messages, tools=None, **_kwargs):
        return StubResponse(
            stop_reason="tool_use",
            tool_calls=[
                StubToolCall(
                    id=f"tc-{next(_tc_counter)}",
                    name="nonexistent",  # registry returns None → tool error, no completion
                    arguments={},
                )
            ],
        )


state = make_run_state()
result = await run_with(state, NeverFinishesModel(), max_turns=3)

print(f"turns           : {result.turns}")
print(f"content         : {result.content}")
print("completion_payload:")
for k, v in state.completion_payload.items():
    print(f"  {k:8s}: {v!r}")

turns           : 3
content         : None
completion_payload:
  status  : 'failed'
  summary : 'Turn limit reached before task completed.'
  error   : 'max_turns'


In [16]:
# The breach emits BOTH loop.max_turns and loop.completed. Inspect the event tail.
tail = [e for e in state.event_bus.events if e.type in {"loop.max_turns", "loop.completed"}]
for evt in tail:
    print(f"[{evt.type}]")
    for k, v in evt.data.items():
        print(f"  {k:11s}: {v!r}")
    print()

[loop.max_turns]
  turns_used : 3
  max_turns  : 3

[loop.completed]
  reason     : 'max_turns'
  cost_usd   : 0.0
  tokens     : {'input': 15, 'output': 9, 'total': 24}



The synthetic payload is the same shape as a payload the agent itself would have produced if it
had called `task_complete(status='failed', summary='...', error='max_turns')`. From the outer
caller's perspective the run failed for `max_turns` — there's no "did the loop crash?" branch.

## 8. `max_cost_usd` enforcement

`RunState.max_cost_usd` is the per-run cost ceiling, in USD. Where it's checked:

- At the **top of every turn**, before calling `model.invoke()`.
- The check is `state.cost_usd >= state.max_cost_usd`. **Greater-or-equal** — once you've hit the
  cap you stop, you don't get one more free turn.
- `state.cost_usd` is accumulated via `_accumulate_usage` after every model response, summing
  `response.cost_usd or 0.0`. So the cap is enforced *after* the response that pushed you over.

Note: `run()` does not currently take `max_cost_usd` as a kwarg — it's set on `RunState` by the
outer driver (e.g. `arcagent`) before the strategy runs. Below we drive `react_loop` directly
with a state that has the cap set, mirroring how arcrun's own tests exercise this path.

In [17]:
# Turn 1 returns a $5 response. Cap is $3. After the response, cost_usd = 5.0.
# At the top of turn 2 the loop sees 5.0 >= 3.0 and terminates with error='max_cost'.
model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            cost_usd=5.0,
            tool_calls=[
                StubToolCall(id="tc-1", name="nonexistent", arguments={}),
            ],
        ),
        StubResponse(stop_reason="end_turn", content="this never runs"),
    ]
)

state = make_run_state(max_cost_usd=3.0)
result = await run_with(state, model, max_turns=10)

print(f"turns          : {result.turns}")
print(f"cost_usd accum : {state.cost_usd}")
print(f"max_cost_usd   : {state.max_cost_usd}")
print()
print("completion_payload:")
for k, v in state.completion_payload.items():
    print(f"  {k:8s}: {v!r}")

turns          : 1
cost_usd accum : 5.0
max_cost_usd   : 3.0

completion_payload:
  status  : 'failed'
  summary : 'Cost limit reached before task completed.'
  error   : 'max_cost'


In [18]:
# The breach emits loop.completed with reason='max_cost' and the running cost.
# (loop.max_turns is NOT emitted in this case — it's a different code path.)
for evt in state.event_bus.events:
    if evt.type == "loop.completed":
        print(f"[{evt.type}]")
        for k, v in evt.data.items():
            print(f"  {k:8s}: {v!r}")

[loop.completed]
  reason  : 'max_cost'
  cost_usd: 5.0
  tokens  : {'input': 5, 'output': 3, 'total': 8}


In [19]:
# Edge case: cost cap is hit exactly. The check is >=, not >, so the loop stops.
model_exact = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            cost_usd=2.0,
            tool_calls=[StubToolCall(id="tc-1", name="nonexistent", arguments={})],
        ),
        StubResponse(stop_reason="end_turn", content="never reached"),
    ]
)

state_exact = make_run_state(max_cost_usd=2.0)  # exactly equal at end of turn 1
result_exact = await run_with(state_exact, model_exact, max_turns=10)

print(f"cost_usd  = {state_exact.cost_usd}")
print(f"cap       = {state_exact.max_cost_usd}")
print(f"terminated by: {state_exact.completion_payload['error']}")
print(f"turns     = {result_exact.turns}")

cost_usd  = 2.0
cap       = 2.0
terminated by: max_cost
turns     = 1


Two boundary observations from the source:

- The cost check fires **before** turn 2, not in the middle of turn 1. So a single very
  expensive response that overshoots the cap doesn't get "refunded" — it's already been paid for,
  but you get no further turns.
- If `max_cost_usd is None` (the default), the check is skipped entirely. Setting `max_cost_usd=0.0`
  means *any* non-zero cost terminates after the first turn — useful for dry-run tests.

## 9. `make_budget_breach_args` — the canonical breach vocabulary

When the loop synthesises a budget-breach payload it needs to pick a `summary` and an `error`
code. `make_budget_breach_args(reason=...)` is the single function that mints that pair, so every
site in the codebase that fabricates a synthetic terminator emits the same vocabulary.

It takes one keyword argument — `reason`, restricted by
`BudgetBreachReason = Literal["max_turns", "max_cost", "max_tokens", "runaway_loop", "error_cascade"]` —
and returns a fully-validated `TaskCompleteArgs` instance. The first three are the token/cost/turn
caps; the last two are the SPEC-043 runaway-loop and error-cascade circuit breakers. Use it from
outer code that wants to shape a payload identically to what the loop would have produced (e.g. when
*you* enforce a budget around `run()` instead of letting `react_loop` enforce it).

In [20]:
from typing import get_args

from arcrun.builtins.task_complete import BudgetBreachReason

for reason in get_args(BudgetBreachReason):
    args = make_budget_breach_args(reason=reason)
    print(f"reason={reason!r}")
    print(f"  status : {args.status}")
    print(f"  summary: {args.summary}")
    print(f"  error  : {args.error}")
    print()

reason='max_turns'
  status : failed
  summary: Turn limit reached before task completed.
  error  : max_turns

reason='max_cost'
  status : failed
  summary: Cost limit reached before task completed.
  error  : max_cost

reason='max_tokens'
  status : failed
  summary: Token limit reached before task completed.
  error  : max_tokens

reason='runaway_loop'
  status : failed
  summary: Repeated identical tool call detected — halted as a runaway loop.
  error  : runaway_loop

reason='error_cascade'
  status : failed
  summary: Consecutive tool failures exceeded the cascade threshold — halted.
  error  : error_cascade



In [21]:
# The helper is restrictive on purpose — the BudgetBreachReason Literal prevents drift.
# An unknown reason has no entry in the summary table, so it raises KeyError at runtime;
# typed callers never reach that, because mypy/pyright reject anything outside the Literal.
try:
    make_budget_breach_args(reason="timeout")  # type: ignore[arg-type]
except KeyError as exc:
    print("At runtime an unknown reason raises:")
    print(f"  KeyError: {exc}")
    print("Typed callers cannot reach this — reason is restricted to the five values above.")

At runtime an unknown reason raises:
  KeyError: 'timeout'
Typed callers cannot reach this — reason is restricted to the five values above.


## 10. Composing with strategies — ReAct vs CodeExec

`task_complete` is **strategy-agnostic**. The terminator mechanism lives in `react_loop`, which
is the underlying engine for both the `react` strategy and the `code` strategy (CodeExec
augments the system prompt and then delegates to `react_loop`). Whichever strategy you select,
the same payload contract holds.

Below we re-run the success scenario through `CodeExecStrategy` to confirm. Note we still
register `task_complete` — strategies don't auto-add it; the caller chooses which terminators
to expose.

In [22]:
# Same model script, run via the CodeExec strategy. The terminator works identically.
from arcrun._messages import system_message

model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            tool_calls=[
                StubToolCall(
                    id="tc-1",
                    name="task_complete",
                    arguments={"status": "success", "summary": "computed via code"},
                )
            ],
        ),
    ]
)

state = make_run_state()
# CodeExec needs a system message at index 0 to splice its prefix onto.
state.messages.insert(0, system_message("You are a Python-first solver."))

sandbox = Sandbox(SandboxConfig(), state.event_bus)
code_strategy = CodeExecStrategy()
result = await code_strategy(model, state, sandbox, max_turns=5)

print(f"strategy.name      : {code_strategy.name}")
print(f"completion status  : {state.completion_payload['status']}")
print(f"result.content     : {result.content}")
print(f"result.turns       : {result.turns}")
print()
print("CodeExec emitted the same loop.completed event:")
for evt in state.event_bus.events:
    if evt.type == "loop.completed":
        print(f"  {dict(evt.data)}")

strategy.name      : code
completion status  : success
result.content     : computed via code
result.turns       : 1

CodeExec emitted the same loop.completed event:
  {'status': 'success', 'summary': 'computed via code'}


**Mid-run completion is fine in either strategy.** The check happens after every model response
— if `signals_completion` fires on turn 1, the loop exits on turn 1. There's no "minimum
turns" or "must use the code strategy first" rule. The agent is free to terminate as soon as
it has a valid answer.

## 11. Patterns — outer callers consuming the payload

What does an `arcagent` (or any outer driver) actually do with the payload? Three canonical
branches.

### Pattern A: the simple dispatch

```python
match payload['status']:
    case 'success':
        return Result.ok(payload)
    case 'partial':
        return Result.handoff(payload)
    case 'failed':
        return Result.error(payload)
```

### Pattern B: budget-aware retry

```python
if payload['status'] == 'failed' and payload.get('error') == 'max_turns':
    # Bump the budget and retry once.
    return await run(model, tools, system, task, max_turns=2 * max_turns)
```

### Pattern C: human-in-the-loop hand-off

```python
if payload['status'] == 'partial' and payload.get('next_steps'):
    notify_human(payload['summary'], payload['next_steps'], payload.get('artifacts', []))
    return Result.pending_human(payload)
```

All three are pure functions of the payload — no parsing, no scraping. That's the value: the
boundary is structured, and the outer code is small.

In [23]:
# Implement Pattern A as a function the rest of this section uses.
from typing import Literal

Outcome = Literal["ok", "handoff", "error"]


def dispatch(payload: dict | None) -> tuple[Outcome, str]:
    """Map a completion payload to an outer-caller outcome + a one-line message."""
    if payload is None:
        return ("error", "no completion payload — run did not terminate cleanly")
    status = payload["status"]
    summary = payload["summary"]
    error = payload.get("error")
    if status == "success":
        return ("ok", summary)
    if status == "partial":
        steps = payload.get("next_steps") or []
        return ("handoff", f"{summary} (next: {len(steps)} step(s))")
    # status == 'failed'
    if error in {"max_turns", "max_cost"}:
        return ("error", f"BUDGET BREACH ({error}): {summary}")
    return ("error", f"AGENT FAILED ({error or 'no_code'}): {summary}")


# Drive every status through the dispatcher.
scenarios = [
    {"status": "success", "summary": "all done"},
    {"status": "partial", "summary": "2 of 3 done", "next_steps": ["fetch data"]},
    {"status": "failed", "summary": "tool said no", "error": "permission_denied"},
    {"status": "failed", "summary": "Turn limit reached.", "error": "max_turns"},
    {"status": "failed", "summary": "Cost limit reached.", "error": "max_cost"},
    None,
]

for s in scenarios:
    outcome, msg = dispatch(s)
    print(f"[{outcome:7s}] {msg}")

[ok     ] all done
[handoff] 2 of 3 done (next: 1 step(s))
[error  ] AGENT FAILED (permission_denied): tool said no
[error  ] BUDGET BREACH (max_turns): Turn limit reached.
[error  ] BUDGET BREACH (max_cost): Cost limit reached.
[error  ] no completion payload — run did not terminate cleanly


In [24]:
# Pattern B in code: budget-aware retry. We synthesise the breach with the helper,
# inspect the error, and decide whether to retry.
import asyncio


async def run_with_retry(
    *,
    model_factory,
    initial_max_turns: int,
    retry_budget_multiplier: int = 2,
) -> dict:
    state = make_run_state()
    await run_with(state, model_factory(), max_turns=initial_max_turns)
    payload = state.completion_payload
    if payload["status"] == "failed" and payload.get("error") == "max_turns":
        new_max = initial_max_turns * retry_budget_multiplier
        retry_state = make_run_state()
        await run_with(retry_state, model_factory(), max_turns=new_max)
        return {
            "attempt": "retry",
            "final_payload": retry_state.completion_payload,
            "new_max_turns": new_max,
        }
    return {"attempt": "first", "final_payload": payload, "new_max_turns": initial_max_turns}


# Factory that produces a fresh model that completes on turn 5 (so max_turns=3 fails, max_turns=6 succeeds).
def slow_completer():
    return StubModel(
        [
            StubResponse(
                stop_reason="tool_use",
                tool_calls=[StubToolCall(id=f"tc-{i}", name="nonexistent", arguments={})],
            )
            for i in range(4)
        ]
        + [
            StubResponse(
                stop_reason="tool_use",
                tool_calls=[
                    StubToolCall(
                        id="tc-final",
                        name="task_complete",
                        arguments={"status": "success", "summary": "eventually got there"},
                    )
                ],
            )
        ]
    )


outcome = await run_with_retry(
    model_factory=slow_completer,
    initial_max_turns=3,
    retry_budget_multiplier=3,
)

print(f"attempt taken      : {outcome['attempt']}")
print(f"final status       : {outcome['final_payload']['status']}")
print(f"final summary      : {outcome['final_payload']['summary']}")
print(f"effective max_turns: {outcome['new_max_turns']}")
_ = asyncio  # silence unused import

attempt taken      : retry
final status       : success
final summary      : eventually got there
effective max_turns: 9


In [25]:
# Pattern C: human hand-off. We capture every artifact + next-step into a structured ticket
# the outer system would file in (Slack, Jira, your queue of choice).
def file_handoff_ticket(payload: dict) -> dict | None:
    if payload["status"] != "partial":
        return None
    return {
        "ticket_type": "agent_handoff",
        "title": payload["summary"],
        "attachments": payload.get("artifacts") or [],
        "todo": payload.get("next_steps") or [],
    }


model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            tool_calls=[
                StubToolCall(
                    id="tc-1",
                    name="task_complete",
                    arguments={
                        "status": "partial",
                        "summary": "wrote 80% of the proposal — pricing section needs human review",
                        "artifacts": ["/drafts/proposal.md"],
                        "next_steps": [
                            "have finance approve the pricing tier",
                            "sign off on legal terms",
                        ],
                    },
                )
            ],
        ),
    ]
)
state = make_run_state()
await run_with(state, model)

ticket = file_handoff_ticket(state.completion_payload)
print("ticket filed:")
for k, v in ticket.items():
    print(f"  {k}: {v}")

ticket filed:
  ticket_type: agent_handoff
  title: wrote 80% of the proposal — pricing section needs human review
  attachments: ['/drafts/proposal.md']
  todo: ['have finance approve the pricing tier', 'sign off on legal terms']


### One more: detecting a non-payload run

If you ever see `state.completion_payload is None` after a run, the loop returned without
completing — the most common cause is `cancel_event` being set externally. Treat that as a
*third-party termination*, not as success or failure. The dispatcher above already handles it.

In [26]:
# Show that a cancelled run leaves completion_payload = None.
model = StubModel(
    [
        StubResponse(
            stop_reason="tool_use",
            tool_calls=[StubToolCall(id="tc-1", name="nonexistent", arguments={})],
        )
    ]
)
state = make_run_state()
state.cancel_event.set()  # cancel before the loop even starts
result = await run_with(state, model, max_turns=5)

print(f"completion_payload : {state.completion_payload}")
print(f"turns              : {result.turns}")
print(f"content            : {result.content}")
print()
print("dispatcher result:")
outcome, msg = dispatch(state.completion_payload)
print(f"  [{outcome}] {msg}")

completion_payload : None
turns              : 0
content            : None

dispatcher result:
  [error] no completion payload — run did not terminate cleanly


## 12. Summary

`task_complete` and the budget caps together give arcrun a **single, structured terminator**
for every run. The contract is simple:

- `state.completion_payload` is set on every clean termination — agent-driven or budget-driven.
- The shape is always `{status, summary, [artifacts], [next_steps], [error]}`.
- `status` is one of `success`, `partial`, `failed` — those three values cover every outcome.
- `error` is the structured code: agent-supplied for explicit failures, or one of `max_turns` /
  `max_cost` for budget breaches.
- `make_budget_breach_args(reason=...)` is the canonical helper that mints the breach payload —
  the loop uses it implicitly; outer code can use it explicitly when wrapping `run()` with its
  own enforcement.
- The terminator mechanism is generic: any tool can opt in via `Tool.signals_completion=True`.
- The same contract applies under both `react` and `code` strategies — `code` delegates to
  `react_loop`, so the termination code path is shared.

**API surface covered** (every public symbol from PLAN.md §5.2.7):

| Symbol                       | Where shown                                                        |
|------------------------------|--------------------------------------------------------------------|
| `make_task_complete_tool`    | §2 (mechanism), §4 (success), §5/§6 (partial/failed), §10 (CodeExec)|
| `TaskCompleteArgs`           | §3 (full schema), §4–§6 (instances embedded in tool calls)         |
| `TaskStatus`                 | §3 (Literal type, exported)                                        |
| `make_budget_breach_args`    | §9 (canonical helper)                                              |
| `max_turns` (kwarg of `run`) | §7 (enforcement, synthetic payload, `loop.max_turns` event)        |
| `max_cost_usd` (`RunState`)  | §8 (enforcement, ≥ check, edge cases)                              |
| `loop.completed` event       | §4, §7, §8, §10 (every termination path)                           |
| `signals_completion` flag    | §2, §4 (mechanism explanation)                                     |

**Next:** see `arcrun/07-event-chain-verification.ipynb` for the deep dive on the tamper-evident
event chain that records every turn / tool call / completion described above. See
`arcrun/01-core-react.ipynb` for the broader tour of the loop, sandbox, registry, and steering.